# Figure 7 and Figure S8: pPRH task-state separation

Main panels **7b–e and 7g–j** and supplementary **S8b–d and S8f–h**. Historical S4 separation filenames correspond to current S8.

This notebook follows **Xu et al.**. Data loading, model fitting, panel selection and plotting are shown below as executable steps. Shared scientific methods remain in `src/`; this notebook does not delegate the figure to `run_figure()`.

Start with the separate manuscript source-data files listed in `data/required_files.csv`. These are analysis source tables and frozen map/embedding arrays, rather than raw movies. The aligned-data computation examples are in `00_recompute_metrics.ipynb`.

Run cells from top to bottom. `REFIT_STATISTICS=True` refits the manuscript models; set it to `False` to use the frozen estimates. Outputs are saved separately from inputs.

## 1. Paths and execution parameters

Set `PPRH_DATA_DIR` or edit `DATA_ROOT`. The default refits statistical models. No raw neural resampling is triggered here.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "main.py").is_file() and (p / "src/paper.py").is_file())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import SVG, Image, display
from paper_io import read_table, save_panel as _save_panel, write_tables, p_symbol, write_run_record

DATA_ROOT = Path(os.environ.get("PPRH_DATA_DIR", str(ROOT / "data/input")))
OUTPUT_ROOT = Path(os.environ.get("PPRH_OUTPUT_DIR", str(ROOT / "outputs/notebooks")))
REFIT_STATISTICS = os.environ.get("PPRH_REFIT_STATISTICS", "1") == "1"
FORMATS = ("svg", "png")
DPI = 300

data_root = DATA_ROOT
output = OUTPUT_ROOT / "Fig7"
refit, formats, dpi = REFIT_STATISTICS, FORMATS, DPI
output.mkdir(parents=True, exist_ok=True)
if not data_root.is_dir():
    raise FileNotFoundError("Supply the separate manuscript inputs through DATA_ROOT; see data/README.md.")


## 2. Analysis contract

Separate Figure-8 route, T-maze sample/choice and T-maze within-phase route contrasts are fitted. Direct C/P models retain lateral-deviation adjustment. Day, mouse and spatial section remain explicit. Quantified manifold separation uses the same frozen 50-neuron, 15-dimensional, 100-repeat/100-shuffle method. The plotting inputs are shuffle-standardized source metrics; raw resampling is not rerun by the model-fit cells.

## 3. Notebook functions

The export function saves and previews each panel. Where needed, the source-table loader or annotation helper is defined here in full. Numerical estimators and shared plotting functions are imported in the relevant analysis cells.

In [ ]:
def save_panel(fig, output_root, panel, description, **kwargs):
    """Save one panel with the shared export method, then preview its result."""
    paths = _save_panel(fig, output_root, panel, description, **kwargs)
    svg = next((p for p in paths if p.endswith(".svg")), None)
    png = next((p for p in paths if p.endswith(".png")), None)
    if svg:
        display(SVG(filename=svg))
    elif png:
        display(Image(filename=png))
    return paths


## 4. Source data, statistics and panel-by-panel plots

Each panel cell names its source, contrast and display parameters. Use the current panel names below; historical filenames remain unchanged for traceability. Model fitting can take longer than figure rendering.

### Shared methods used below

In [ ]:
from maze_analysis import fig7_figure8_pprh as f8, fig7_figure8_pprh_plotting as p8


### Shared methods used below

In [ ]:
from maze_analysis import fig7_s4_tmaze_pprh as tm, fig7_s4_tmaze_pprh_plotting as pt


### Shared methods used below

In [ ]:
from maze_analysis import figs4_ad_tmaze_pv as pv, figs4_ad_tmaze_pv_plotting as ppv


### Read the source data

In [ ]:
from figure_helpers import manifold_heatmaps as hm

exports = []

table8 = read_table(data_root, 'fig7_figure8/source_data/05_Fig7BE_longitudinal_model_table_D1-D5.csv')

stats8 = read_table(data_root, 'fig7_figure8/statistics/01_all_lmm_contrasts.csv')


In [ ]:
display(table8.head())


### Fit the manuscript models or retain the frozen estimates

In [ ]:
if refit:
    result = f8.fit_figure8_pprh_models(table8)
    stats8 = result.contrasts
    write_tables({'figure8_pprh_refitted_estimands': stats8}, Path(output) / 'statistics')


In [ ]:
display(stats8.head())


### Fig7 b — prepare panel inputs

In [ ]:
annotations = f8.build_annotation_payloads(stats8)

bins = read_table(data_root, 'fig7_figure8/source_data/01_Fig7A_pPRH_high_dimensional_PV_bins_D1-D5.csv')


### Fig7 b — draw and export

In [ ]:
fig = p8.plot_pprh_pv_heatmap(bins, vmin=-0.5, vmax=10.5)[0]

exports += save_panel(fig, output, 'Fig7_b', 'PV_route_heatmap', formats=formats, dpi=dpi)


### Fig7 c — prepare panel inputs

In [ ]:
panel, metric, label = ('c', 'high_dimensional_pv', 'z-scored PV separation')


### Fig7 c — draw and export

In [ ]:
fig = p8.plot_section_curves(table8, f8.build_group_day_summary(table8), metric=metric, ylabel=label, stat_annotations=annotations)[0]

exports += save_panel(fig, output, f'Fig7_{panel}', 'route_state_separation', formats=formats, dpi=dpi)


### Fig7 e — prepare panel inputs

In [ ]:
panel, metric, label = ('e', 'neural_manifold', 'z-scored d′')


### Fig7 e — draw and export

In [ ]:
fig = p8.plot_section_curves(table8, f8.build_group_day_summary(table8), metric=metric, ylabel=label, stat_annotations=annotations)[0]

exports += save_panel(fig, output, f'Fig7_{panel}', 'route_state_separation', formats=formats, dpi=dpi)


### Read the source data

In [ ]:
ttable = read_table(data_root, 'fig7_tmaze/source_data/08_CP_longitudinal_LMM_input.csv')

tstats = read_table(data_root, 'fig7_tmaze/statistics/Fig7_S4_LMM_all_contrasts.csv')


In [ ]:
display(ttable.head())


### Fit the manuscript models or retain the frozen estimates

In [ ]:
if refit:
    result = tm.fit_cp_longitudinal_models(ttable)
    tstats = result.contrasts
    write_tables({'tmaze_pprh_refitted_estimands': tstats}, Path(output) / 'statistics')


In [ ]:
display(tstats.head())


### Fig7 g — prepare panel inputs

In [ ]:
tann = tm.build_annotation_payloads(tstats)

bins = read_table(data_root, 'fig7_tmaze/source_data/05_Fig7F_high_dimensional_PV_spatial_bin.csv')


### Fig7 g — draw and export

In [ ]:
fig = pt.plot_fig7f_pv_heatmap(bins, vmin=-0.5, vmax=2.5)[0]

exports += save_panel(fig, output, 'Fig7_g', 'PV_phase_heatmap', old_label='F', formats=formats, dpi=dpi)


### Fig7 h — prepare panel inputs

In [ ]:
panel, old, metric, contrast = ('Fig7_h', '7G', 'high_dimensional_pv_separation', 'sample_vs_choice')


### Fig7 h — draw and export

In [ ]:
fig = pt.plot_cp_section_curves(ttable, tm.build_group_day_summary(ttable), metric=metric, contrast=contrast, panel=old, annotations=tann)[0]

exports += save_panel(fig, output, panel, 'task_state_separation', old_label=old, formats=formats, dpi=dpi)


### Fig7 j — prepare panel inputs

In [ ]:
panel, old, metric, contrast = ('Fig7_j', '7I', 'neural_manifold_separation', 'sample_vs_choice')


### Fig7 j — draw and export

In [ ]:
fig = pt.plot_cp_section_curves(ttable, tm.build_group_day_summary(ttable), metric=metric, contrast=contrast, panel=old, annotations=tann)[0]

exports += save_panel(fig, output, panel, 'task_state_separation', old_label=old, formats=formats, dpi=dpi)


### FigS8 d — prepare panel inputs

In [ ]:
panel, old, metric, contrast = ('FigS8_d', 'S4E', 'neural_manifold_separation', 'sample_left_vs_right')


### FigS8 d — draw and export

In [ ]:
fig = pt.plot_cp_section_curves(ttable, tm.build_group_day_summary(ttable), metric=metric, contrast=contrast, panel=old, annotations=tann)[0]

exports += save_panel(fig, output, panel, 'task_state_separation', old_label=old, formats=formats, dpi=dpi)


### FigS8 h — prepare panel inputs

In [ ]:
panel, old, metric, contrast = ('FigS8_h', 'S4F', 'neural_manifold_separation', 'choice_left_vs_right')


### FigS8 h — draw and export

In [ ]:
fig = pt.plot_cp_section_curves(ttable, tm.build_group_day_summary(ttable), metric=metric, contrast=contrast, panel=old, annotations=tann)[0]

exports += save_panel(fig, output, panel, 'task_state_separation', old_label=old, formats=formats, dpi=dpi)


### Read the source data

In [ ]:
raw = read_table(data_root, 'pprh_manifold_heatmaps/source_data/01_pPRH_neural_manifold_mouse_day_bin_rows.csv')


### Fig7 d — prepare panel inputs

In [ ]:
spec, panel = (hm.PANEL_SPECS[0], 'Fig7_d')

selected = raw[raw.panel_key.eq(spec.key)].copy()


### Fig7 d — draw and export

In [ ]:
fig = hm.render_panel(selected, spec)

exports += save_panel(fig, output, panel, 'manifold_heatmap', formats=formats, dpi=dpi)


### Fig7 i — prepare panel inputs

In [ ]:
spec, panel = (hm.PANEL_SPECS[1], 'Fig7_i')

selected = raw[raw.panel_key.eq(spec.key)].copy()


### Fig7 i — draw and export

In [ ]:
fig = hm.render_panel(selected, spec)

exports += save_panel(fig, output, panel, 'manifold_heatmap', formats=formats, dpi=dpi)


### Read the source data

In [ ]:
ptable = read_table(data_root, 'supp_pv/source_data/08_FigS4BD_longitudinal_LMM_input.csv')

pstats = read_table(data_root, 'supp_pv/statistics/FigS4AD_LMM_all_contrasts.csv')


In [ ]:
display(ptable.head())


### Fit the manuscript models or retain the frozen estimates

In [ ]:
if refit:
    pstats = pv.fit_longitudinal_models(ptable).contrasts
    write_tables({'route_pv_refitted_estimands': pstats}, Path(output) / 'statistics')


In [ ]:
display(pstats.head())


### Read the source data

In [ ]:
pann = pv.build_annotation_payloads(pstats)

pbins = read_table(data_root, 'supp_pv/source_data/06_FigS4AC_highdim_PV_spatial_bin.csv')


### FigS8 b — prepare panel inputs

In [ ]:
phase, heatpanel, curvepanel, oldh, oldc = ('sample_left_vs_right', 'b', 'c', 'S4A', 'S4B')


### FigS8 b — draw and export

In [ ]:
fig = ppv.plot_pprh_spatial_heatmap(pbins, contrast=phase, panel=oldh, vmin=-1, vmax=1.5)[0]

exports += save_panel(fig, output, f'FigS8_{heatpanel}', 'route_PV_heatmap', old_label=oldh, formats=formats, dpi=dpi)


### FigS8 c — draw and export

In [ ]:
fig = ppv.plot_cp_section_curves(ptable, pv.build_group_day_summary(ptable), contrast=phase, panel=oldc, annotations=pann)[0]

exports += save_panel(fig, output, f'FigS8_{curvepanel}', 'route_PV_separation', old_label=oldc, formats=formats, dpi=dpi)


### FigS8 f — prepare panel inputs

In [ ]:
phase, heatpanel, curvepanel, oldh, oldc = ('choice_left_vs_right', 'f', 'g', 'S4C', 'S4D')


### FigS8 f — draw and export

In [ ]:
fig = ppv.plot_pprh_spatial_heatmap(pbins, contrast=phase, panel=oldh, vmin=-1, vmax=1.5)[0]

exports += save_panel(fig, output, f'FigS8_{heatpanel}', 'route_PV_heatmap', old_label=oldh, formats=formats, dpi=dpi)


### FigS8 g — draw and export

In [ ]:
fig = ppv.plot_cp_section_curves(ptable, pv.build_group_day_summary(ptable), contrast=phase, panel=oldc, annotations=pann)[0]

exports += save_panel(fig, output, f'FigS8_{curvepanel}', 'route_PV_separation', old_label=oldc, formats=formats, dpi=dpi)


## 5. Export record

Panel files and any refitted model tables are saved under the figure output directory. The record identifies the selected manuscript workflow and whether statistics were refitted.

In [ ]:
write_run_record(output / "run.json", figure="7", refit=refit, exports=exports)
display(pd.DataFrame({"exported_file": [str(Path(p).relative_to(output)) for p in exports]}))
